# Volume 10, Chapter 10: Ergodic theory (Python companion)

Exercises first, solutions at the end. We see why the doubling map cannot be simulated in floating point, check Kac's lemma, measure the continued fraction digits of random numbers and of $\pi$, and look for long arithmetic progressions in sets that avoid them. Two computer science applications: **Markov chain Monte Carlo**, where the ergodic theorem is the reason the method works and the mixing time is its price, and **quasi-Monte Carlo** integration with Weyl sequences. Requires `numpy`.

In [ ]:
import math, random
from fractions import Fraction
import numpy as np
rng = np.random.default_rng(100)
random.seed(100)

## Exercises

**P10.1 (★) The doubling map on a computer.** Iterate $x\mapsto2x\bmod1$ in double precision from $x_0=$ `rng.random()`, and compute the proportion of the first $10^4$ iterates in $[0,\frac12)$. Birkhoff's theorem predicts $\frac12$ for almost every $x_0$. What happens, and why? Redo it correctly with exact random bits.

**P10.2 (★) Kac's lemma.** In $10^6$ fair random bits, measure the mean gap between consecutive occurrences of the patterns `11`, `10`, `111`, `1011`, and compare with $2^\ell$. Measure also the mean waiting time for the first occurrence of each pattern (Volume 3's formula: the sum of $2^k$ over the overlaps $k$ of the pattern with itself).

**P10.3 (★★) The digits of random numbers and of $\pi$.** For $200$ random numbers given by $20\,000$ random bits each, compute the continued fraction digits (stop at $n=5000$). Compare (a) the digit frequencies with $\log_2(1+\frac1{k(k+2)})$, (b) the geometric mean with Khinchin's $K=2.685452\ldots$ (compute $K$ from its product, with a tail correction), (c) $\frac1n\log q_n$ with $\frac{\pi^2}{12\log2}$. Then compute $\pi$ to $6000$ decimals with Machin's formula in integer arithmetic, and test its first $5000$ digits the same way.

**P10.4 (★★) CS application: MCMC and mixing.** The Metropolis algorithm samples from a density $\pi$ by proposing $y=x+sZ$ and accepting with probability $\min(1,\pi(y)/\pi(x))$; $\pi$ is invariant for this Markov chain, and the ergodic theorem (for Markov chains, a consequence of Birkhoff's theorem applied to the shift on paths) makes time averages converge. Take $\pi$ the mixture $\frac12\mathcal N(-3,1)+\frac12\mathcal N(3,1)$ and estimate $P(X>0)=\frac12$ with $10^5$ steps, for step sizes $s=0.5,2.5,10$. Report the acceptance rate, the number of jumps between the two modes, the integrated autocorrelation time of $\mathbf 1\{X>0\}$, and the estimate with its error bar (batch means).

**P10.5 (★★) Sets without three-term progressions.** (a) Compute $r_3(N)$, the largest size of a subset of $\{1,\dots,N\}$ with no three-term progression, for $N\le40$ by a branch-and-bound search. (b) Compare the density of the set of integers whose base-$3$ digits are $0$ or $1$ (Erdős–Turán) with that of Behrend's construction (1946) for $N=10^6$ and $N=10^8$: the integers whose base-$d$ digits are all $<d/2$ and whose digit vectors lie on one sphere $\sum x_i^2=r$ (choose $d$ and $r$ to maximize the size). Roth's theorem says all these densities tend to $0$; how fast?

**P10.6 (★★★) CS application: quasi-Monte Carlo.** By Weyl's theorem (Exercise 10.5), the points $\{n\alpha\}$ are equidistributed for irrational $\alpha$. Compute the star discrepancy $D_N=\sup_t|\frac1N\#\{n<N:\{n\alpha\}<t\}-t|$ for $N=10^2,\dots,10^6$ and $\alpha=\frac{\sqrt5-1}2$, $\sqrt2-1$, $e-2$, $\pi-3$, and compare with $N$ uniform random points. Then integrate $f(t)=e^{\sin2\pi t}$ (exact value $I_0(1)$) with the golden-ratio points and with random points: how does the error decrease with $N$?

## Solutions

### P10.1
**How to find it.** A double has $53$ significant bits, and each doubling shifts one bit out. After about $53$ steps the number is exactly $0$ and stays there.

In [ ]:
x = rng.random(); orbit = []
for _ in range(10**4):
    orbit.append(x); x = (2 * x) % 1.0
orbit = np.array(orbit)
print("first zero at step", int(np.argmax(orbit == 0)), "; proportion in [0,1/2):", (orbit < 0.5).mean())
bits = rng.integers(0, 2, 10**4 + 60)                  # x = 0.b1 b2 b3 ... in binary
first_bit = bits[:10**4]                                  # D^n x < 1/2  iff  b_{n+1} = 0
print("with exact bits: proportion in [0,1/2) =", (first_bit == 0).mean())

In floating point, the orbit reaches $0$ after about $53$ steps and the "time average" is close to $1$: the computer simulates a system whose only invariant measure is $\delta_0$. With exact random bits the doubling map is the shift, and the proportion is close to $\frac12$. Chaotic maps amplify rounding errors exponentially, so a long simulated orbit is never a true orbit; the shadowing lemma of hyperbolic dynamics says that for some maps it is close to *another* true orbit, which is why simulations of chaos are still informative.

### P10.2
**How to find it.** Kac: the mean gap between occurrences of a pattern of length $\ell$ is $1/P(\text{pattern})=2^\ell$, whatever its overlaps. The waiting time from scratch is $\sum_{k\text{ overlap}}2^k$.

In [ ]:
b = rng.integers(0, 2, 10**6)
s = "".join(map(str, b))
for w in ("11", "10", "111", "1011"):
    pos = [i for i in range(len(s) - len(w) + 1) if s.startswith(w, i)]
    gaps = np.diff(pos)
    waits = []
    for t in range(4000):
        c = rng.integers(0, 2, 400); cs = "".join(map(str, c)); j = cs.find(w)
        if j >= 0: waits.append(j + len(w))
    overl = sum(2**k for k in range(1, len(w) + 1) if w[:k] == w[-k:])
    print(f"{w:5s} mean gap {gaps.mean():7.3f} (Kac: {2**len(w)})   mean first waiting time {np.mean(waits):6.2f} (Volume 3: {overl})")

The gaps average $4$, $4$, $8$, $16$ as Kac predicts, while the first waiting times are $6$, $4$, $14$, $18$: self-overlapping patterns come in clumps, so they are rarer at the start but recur quickly once seen. Both facts are consistent: a long gap after a clump compensates the short gaps inside it.

### P10.3
**How to find it.** The Euclidean algorithm on a random fraction $p/2^{20000}$ gives its digits; a random $20\,000$-bit number agrees with its continued fraction for about $20000\log2/(2\cdot1.1866)\approx5800$ digits, so we stop at $5000$. For $\pi$, Machin's formula $\frac\pi4=4\arctan\frac15-\arctan\frac1{239}$ with integer arithmetic gives $\lfloor10^{6000}\pi\rfloor$.

In [ ]:
def cf_digits(p, q, nmax):
    out = []
    while q and len(out) < nmax:
        a, r = divmod(p, q); out.append(a); p, q = q, r
    return out
LEVY = math.pi**2 / (12 * math.log(2))
logK = sum(math.log2(1 + 1 / (k * (k + 2))) * math.log(k) for k in range(1, 10**6))
logK += (math.log(10**6) + 1) / (10**6 * math.log(2))      # tail: sum_{k>=M} log k /(k^2 log 2)
K = math.exp(logK)
def stats_of(digits):
    a = np.array(digits[1:], dtype=object)                     # drop a_0
    n = len(a)
    q_prev, q = 0, 1
    for x in digits[1:]:
        q_prev, q = q, x * q + q_prev
    geo = math.exp(sum(math.log(int(x)) for x in a) / n)
    return n, geo, math.log(q) / n
counts = np.zeros(11); total = 0; geos = []; levys = []
for r in range(200):
    p = random.getrandbits(20000)
    d = cf_digits(p, 2**20000, 5001)
    for x in d[1:]:
        if x <= 10: counts[x] += 1
    total += len(d) - 1
    n, g, l = stats_of(d); geos.append(g); levys.append(l)
print(" k  frequency   log2(1+1/(k(k+2)))")
for k in range(1, 11):
    print(f"{k:2d}  {counts[k]/total:.5f}    {math.log2(1 + 1/(k*(k+2))):.5f}")
print(f"geometric mean: {np.mean(geos):.4f} +- {np.std(geos)/math.sqrt(200):.4f}   Khinchin K = {K:.6f}")
print(f"(1/n) log q_n : {np.mean(levys):.4f} +- {np.std(levys)/math.sqrt(200):.4f}   Levy = {LEVY:.6f}")

def arctan_inv(x, unity):
    total = term = unity // x; x2 = x * x; n = 1; sign = -1
    while term:
        term //= x2; total += sign * (term // (2 * n + 1)); sign = -sign; n += 1
    return total
unity = 10**6010
pi_scaled = 4 * (4 * arctan_inv(5, unity) - arctan_inv(239, unity))
dpi = cf_digits(pi_scaled, unity, 5001)
print("\npi =", dpi[:12], "...")
n, g, l = stats_of(dpi)
c1 = sum(1 for x in dpi[1:] if x == 1) / n
print(f"pi, {n} digits: frequency of 1 = {c1:.4f} (Gauss 0.4150), geometric mean {g:.4f}, (1/n)log q_n = {l:.4f}")

The frequencies agree with Gauss's law to three decimals ($41.5\%$ of digits are $1$, $17.0\%$ are $2$), the geometric means average to $K\approx2.6855$ within the error bar, and $\frac1n\log q_n$ to Lévy's $1.1866$. The individual geometric means fluctuate by a few percent at $n=5000$: Khinchin's convergence is slow, because of the rare large digits. The continued fraction of $\pi$, $[3;7,15,1,292,1,1,1,2,1,3,1,\dots]$, is indistinguishable from a random one by these statistics, although nobody can prove anything about it.

### P10.4
**How to find it.** With small steps the chain rarely crosses the valley between the modes; with huge steps it rarely accepts. The integrated autocorrelation time $\tau=1+2\sum_{k\ge1}\rho_k$ (summed until the autocorrelation becomes negligible) measures how many steps make one independent sample; the effective sample size is $n/\tau$.

In [ ]:
def logpi(x):
    return np.logaddexp(-0.5 * (x + 3)**2, -0.5 * (x - 3)**2)
def metropolis(s, n=10**5, x0=0.0, rng=rng):
    x = x0; lp = logpi(x); out = np.empty(n); acc = 0
    Z = rng.standard_normal(n); U = np.log(rng.random(n))
    for i in range(n):
        y = x + s * Z[i]; ly = logpi(y)
        if U[i] < ly - lp:
            x, lp = y, ly; acc += 1
        out[i] = x
    return out, acc / n
def iact(f, maxlag=20000):
    f = f - f.mean(); v = f.var()
    if v == 0: return float("inf")
    tau = 1.0
    for k in range(1, maxlag):
        rho = np.dot(f[:-k], f[k:]) / ((len(f) - k) * v)
        if rho < 0.01: break
        tau += 2 * rho
    return tau
for s in (0.5, 2.5, 10.0):
    xs, acc = metropolis(s)
    ind = (xs > 0).astype(float)
    jumps = int(np.sum(np.abs(np.diff(ind))))
    batches = ind.reshape(50, -1).mean(axis=1)
    est, se = ind.mean(), batches.std(ddof=1) / math.sqrt(50)
    tau = iact(ind)
    print(f"s = {s:4.1f}: acceptance {acc:.3f}, mode switches {jumps:5d}, tau = {tau:8.1f}, "
          f"ESS = {len(ind)/tau:7.0f},  P(X>0) = {est:.3f} +- {se:.3f}")

With $s=0.5$ the chain accepts $85\%$ of its proposals but crosses between the modes only about $200$ times in $10^5$ steps: its autocorrelation time is above $1000$, the effective sample size is below $100$, and the error bar is $\pm0.05$. With $s=2.5$ the chain switches modes thousands of times and $\tau$ is about $20$. With $s=10$ only a quarter of the proposals are accepted, but almost every accepted jump is a big one, and here $\tau$ is even smaller, about $10$; much larger steps would eventually lose, since almost all proposals would land in the tails and be rejected. The ergodic theorem guarantees convergence for all three; only the *rate*, governed by how fast the chain mixes, distinguishes a useful sampler from a useless one.

### P10.5
**How to find it.** (a) Search the subsets of $\{1,\dots,N\}$ in decreasing order of the next element to add, pruning when the current size plus the remaining elements cannot beat the best known (and using $r_3(N)\le r_3(N-1)+1$). (b) Behrend: the digit vectors $x\in\{0,\dots,\lceil d/2\rceil-1\}^k$ on a sphere $|x|^2=r$ have no three collinear points in progression, and since the digits are small, adding two numbers never carries; so the corresponding integers contain no progression.

In [ ]:
def r3(N):
    best = [0]
    def ok(S, x):
        return all((2 * a - x) not in S for a in S) and all(((a + x) % 2 == 1) or ((a + x) // 2 not in S) for a in S)
    sys_best = {}
    def search(i, S):
        if len(S) + (N - i + 1) <= best[0]: return
        if i > N:
            best[0] = max(best[0], len(S)); return
        if ok(S, i):
            S.add(i); search(i + 1, S); S.remove(i)
        search(i + 1, S)
    search(1, set())
    return best[0]
vals = [r3(N) for N in range(1, 41)]
print("r_3(N), N = 1..40:", vals)

def ternary01_count(N):
    return sum(1 for m in range(N) if "2" not in np.base_repr(m, 3))
def behrend(N):
    best = (0, None)
    for d in range(3, 60):
        k = int(math.log(N, d))
        if k < 1: continue
        h = (d + 1) // 2                                   # digits 0..h-1
        counts = {}
        for x in np.ndindex(*([h] * k)):
            r = sum(v * v for v in x); counts[r] = counts.get(r, 0) + 1
            if h**k > 2 * 10**5: break
        if h**k > 2 * 10**5: continue
        c = max(counts.values())
        if c > best[0]: best = (c, (d, k))
    return best
for N in (10**6, 10**8):
    k3 = int(math.log(N, 3)); size3 = 2**k3
    b, (d, k) = behrend(N)
    print(f"N = {N:.0e}: digits 0/1 in base 3: {size3} elements (density {size3/N:.2e});  "
          f"Behrend base {d}, {k} digits: {b} elements (density {b/N:.2e})")

The exact values start $1,2,2,3,4,4,4,4,5,5,6,6,7,8,8,8,8,8,8,9,\dots$ and reach $r_3(40)=15$ (a plain exhaustive search over all subsets confirms the first twenty; the sequence is A003002 in the OEIS). The base-$3$ set has density $(2/3)^{\log_3N}=N^{\log_32-1}\approx N^{-0.37}$, which decays polynomially. Behrend's spheres have size $N\exp(-c\sqrt{\log N})$, which eventually beats every power $N^{1-\varepsilon}$; but in our range the search above (limited to small digit boxes) does worse than the base-$3$ set: Behrend's advantage only shows for astronomically large $N$. Roth's theorem says $r_3(N)/N\to0$; the best upper bounds known today (Kelley and Meka, 2023) are of the shape $N\exp(-c(\log N)^{1/12})$, not far from Behrend's lower bound.

### P10.6
**How to find it.** For sorted points $u_1\le\dots\le u_N$, the star discrepancy is $\max_i\max(\frac iN-u_i,u_i-\frac{i-1}N)$. For the golden ratio, Weyl sums are bounded and $D_N=O(\frac{\log N}N)$ (by its continued fraction $[0;1,1,1,\dots]$), while random points have $D_N\approx\frac{0.87}{\sqrt N}$ (Kolmogorov–Smirnov).

In [ ]:
def star_disc(u):
    u = np.sort(u); N = len(u); i = np.arange(1, N + 1)
    return max(np.max(i / N - u), np.max(u - (i - 1) / N))
alphas = {"golden": (math.sqrt(5) - 1) / 2, "sqrt2-1": math.sqrt(2) - 1, "e-2": math.e - 2, "pi-3": math.pi - 3}
print("      N   " + "  ".join(f"{k:>9s}" for k in alphas) + "     random")
for N in (10**2, 10**3, 10**4, 10**5, 10**6):
    n = np.arange(N)
    row = [star_disc((n * a) % 1.0) for a in alphas.values()]
    print(f"{N:8d}  " + "  ".join(f"{v:9.2e}" for v in row) + f"  {star_disc(rng.random(N)):9.2e}")
I0 = sum((0.5)**(2 * j) / math.factorial(j)**2 for j in range(30))   # I_0(1)
print("\n      N   QMC error (golden)   MC error (random)")
for N in (10**2, 10**3, 10**4, 10**5, 10**6):
    u = (np.arange(1, N + 1) * alphas["golden"]) % 1.0
    qmc = abs(np.exp(np.sin(2 * np.pi * u)).mean() - I0)
    mc = abs(np.exp(np.sin(2 * np.pi * rng.random(N))).mean() - I0)
    print(f"{N:8d}   {qmc:12.2e}         {mc:12.2e}")

The Weyl sequences have discrepancy of order $\frac1N$ up to logarithms, about $25$ times smaller than random points at $N=10^4$ and $300$ times at $N=10^6$. The exception is $\pi-3=[0;7,15,1,292,\dots]$, which is no better than random points up to $N=10^4$, because of its large digit $292$: $\pi$ is very close to $\frac{355}{113}$, so the points cluster near $113$ positions for a long while. For the smooth periodic integrand, quasi-Monte Carlo with the golden ratio is spectacular: the error drops, irregularly, faster than $1/N$ (to about $10^{-8}$ at $N=10^6$), since the Weyl sums of each Fourier mode stay bounded and the Fourier coefficients of $e^{\sin2\pi t}$ decay very fast, while Monte Carlo errors decay like $1/\sqrt N$. This is why quasi-Monte Carlo is used for high-precision integration in finance and graphics, in higher-dimensional versions (Kronecker, Halton and Sobol sequences).